## Training the Model

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [1]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # prevent JAX from grabbing all memory upfront

import jax
import jax.numpy as jnp
import flax.nnx as nnx
import grain.python as pygrain
import tiktoken

import optax

In [2]:
from helper import MiniGPT, load_and_preprocess_data, generate_text

In [3]:
maxlen=128
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
num_epochs = 20  # defined here so the DataLoader knows how many epochs to prepare

text_dl, batches_per_epoch = load_and_preprocess_data(
    file_path='TinyStories-1000.txt',
    batch_size=8,
    maxlen=128,
    max_stories=1000,
    num_epochs=num_epochs,
    shuffle=True,
    seed=42
)

Loading data from TinyStories-1000.txt (max 1,000 stories)
Loaded 1,000 stories
Estimated batches per epoch: 125
Created DataLoader with batch_size=8, maxlen=128


In [5]:
model = MiniGPT()

In [6]:
def loss_fn(model, batch):
    inputs, targets = batch
    logits = model(inputs)
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits, targets
    ).mean()
    return loss, logits

In [7]:
total_steps = batches_per_epoch * num_epochs
warmup_steps = max(1, total_steps // 10)  # 10% warmup
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

Total training steps: 2,500
Warmup steps: 250


In [8]:
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=3e-4,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=1e-5
)

## Use an optimizer from optax

In [9]:
optimizer = nnx.Optimizer(
    model,
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.01)
)

In [10]:
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
)

## Define and JIT-compile the training step

In [11]:
@nnx.jit
def train_step(model, optimizer, metrics, batch):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch)

    metrics.update(loss=loss, logits=logits, labels=batch[1])
    optimizer.update(grads)

## Run the training loop

In [ ]:
metrics_history = {'train_loss': []}

prep_target_batch = jax.vmap(
    lambda tokens: jnp.concatenate((tokens[1:], jnp.array([0]))))

for epoch in range(num_epochs):
    step = 0
    for batch in text_dl:
        input_batch = jnp.array(jnp.array(batch).T).astype(jnp.int32)
        target_batch = prep_target_batch(
            jnp.array(jnp.array(batch).T)).astype(jnp.int32)
        print(".", end="")
        train_step(model, optimizer, metrics, (input_batch, target_batch))

        if (step + 1) % 2 == 0:
            for metric, value in metrics.compute().items():
                metrics_history[f'train_{metric}'].append(value)
            metrics.reset()

            current_lr = lr_schedule(step)
            print(f"\nEpoch: {epoch + 1}, Step {step + 1}, Loss: {metrics_history['train_loss'][-1]:.4f}, "
                  f"LR: {current_lr:.2e}")
        step += 1

..
Epoch: 1, Step 2, Loss: 10.9243, LR: 1.20e-06
..
Epoch: 1, Step 4, Loss: 10.9171, LR: 3.60e-06
..
Epoch: 1, Step 6, Loss: 10.9091, LR: 6.00e-06
..
Epoch: 1, Step 8, Loss: 10.9148, LR: 8.40e-06
..
Epoch: 1, Step 10, Loss: 10.9145, LR: 1.08e-05
..
Epoch: 1, Step 12, Loss: 10.8805, LR: 1.32e-05
..
Epoch: 1, Step 14, Loss: 10.8719, LR: 1.56e-05
..
Epoch: 1, Step 16, Loss: 10.8515, LR: 1.80e-05
..
Epoch: 1, Step 18, Loss: 10.8380, LR: 2.04e-05
..
Epoch: 1, Step 20, Loss: 10.8277, LR: 2.28e-05
..
Epoch: 1, Step 22, Loss: 10.7939, LR: 2.52e-05
..
Epoch: 1, Step 24, Loss: 10.7646, LR: 2.76e-05
..
Epoch: 1, Step 26, Loss: 10.7809, LR: 3.00e-05
..
Epoch: 1, Step 28, Loss: 10.7380, LR: 3.24e-05
..
Epoch: 1, Step 30, Loss: 10.6986, LR: 3.48e-05
..
Epoch: 1, Step 32, Loss: 10.6824, LR: 3.72e-05
..
Epoch: 1, Step 34, Loss: 10.6773, LR: 3.96e-05
..
Epoch: 1, Step 36, Loss: 10.6515, LR: 4.20e-05
..
Epoch: 1, Step 38, Loss: 10.6033, LR: 4.44e-05
..
Epoch: 1, Step 40, Loss: 10.5869, LR: 4.68e-05
..
E

In [ ]:
import matplotlib.pyplot as plt
plt.plot(metrics_history['train_loss'])
plt.title('Training Loss — 3 steps, 100 stories')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

### Training loss for 3 epochs, 2,000,000 stories

<img src="training_loss.png" width="570">


## Save model checkpoints

In [ ]:
from pathlib import Path
import orbax

checkpoint_path = Path.cwd() / "small_checkpoint.orbax"

checkpointer = orbax.checkpoint.PyTreeCheckpointer()

checkpointer.save(checkpoint_path, nnx.state(model), force=True)
print(f"Model saved as {checkpoint_path}")